In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import numpy as np

In [2]:
load_dotenv()
engine = create_engine(os.getenv("SUPABASE_DB_URL"))

query = "SELECT * FROM weather_logs_cleaned ORDER BY timestamp ASC;"
df = pd.read_sql(query, con=engine)
df.tail()

,timestamp,temperature_2m,relative_humidity_2m,rain,surface_pressure,precipitation,wind_speed_10m,wind_direction_10m,apparent_temperature,cloud_cover,wind_gusts_10m
3252,2026-09-14 13:00,20.92,1.0,0.0,1007.14,0.0,5.40,301.0,15.97,100.0,15.84
3253,2026-09-14 14:00,21.62,1.0,0.0,1006.97,0.0,4.68,274.0,16.77,99.0,14.40
3254,2026-09-14 15:00,21.62,1.0,0.0,1006.97,0.0,5.76,258.0,16.62,100.0,13.68
3255,2026-09-14 16:00,21.32,1.0,0.0,1006.76,0.0,6.48,298.0,16.21,100.0,13.68
3256,2026-09-14 17:00,20.52,1.0,0.1,1006.92,0.1,3.60,328.0,15.83,100.0,19.80


In [3]:
df['timestamp'] = pd.to_datetime(df['timestamp'])

df['year'] = df['timestamp'].dt.year
df['month'] = df['timestamp'].dt.month
df['day'] = df['timestamp'].dt.day


# Lag features

In [4]:
for lag in [24, 48, 72]:

    # Hőmérséklet
    df[f"temp_lag_{lag}"] = (
        df["temperature_2m"].shift(lag)
    )

    # Páratartalom
    df[f"humidity_lag_{lag}"] = (
        df["relative_humidity_2m"].shift(lag)
    )

    # Légnyomás
    df[f"pressure_lag_{lag}"] = (
        df["surface_pressure"].shift(lag)
    )

    # Szélsebesség
    df[f"wind_lag_{lag}"] = (
        df["wind_speed_10m"].shift(lag)
    )

    # Csapadék
    df[f"rain_lag_{lag}"] = (
        df["rain"].shift(lag)
    )

# Rolling Window Features

In [5]:


# Az elmúlt 24 órában lehullott csapadék
df["rain_sum_24h"] = (
    df["rain"]
    .rolling(24)
    .sum()
)

# Az elmúlt 48 órában lehullott csapadék
df["rain_sum_48h"] = (
    df["rain"]
    .rolling(48)
    .sum()
)

# Az elmúlt 72 órában lehullott csapadék
df["rain_sum_72h"] = (
    df["rain"]
    .rolling(72)
    .sum()
)


# Az elmúlt 24 óra átlagos páratartalma
df["humidity_mean_24h"] = (
    df["relative_humidity_2m"]
    .rolling(24)
    .mean()
)


# Az elmúlt 24 óra átlagos légnyomása
df["pressure_mean_24h"] = (
    df["surface_pressure"]
    .rolling(24)
    .mean()
)


# Légnyomás változása az előző 24 órához képest
df["pressure_delta_24h"] = (
    df["surface_pressure"]
    - df["surface_pressure"].shift(24)
)

# Delta Features

In [6]:
df['temp_delta'] = df['temperature_2m'].diff()
df['pressure_delta'] = df['surface_pressure'].diff()
df['wind_delta'] = df['wind_speed_10m'].diff()
df['rain_delta'] = df['rain'].diff()


# Trigonometrical Features

In [7]:
df['wind_dir_sin'] = np.sin(np.deg2rad(df['wind_direction_10m']))
df['wind_dir_cos'] = np.cos(np.deg2rad(df['wind_direction_10m']))


In [8]:
df['cloud_binary'] = (df['cloud_cover'] > 50).astype(int)
df['cloud_roll_mean_6'] = df['cloud_cover'].rolling(6).mean()


In [9]:
df["rain_sum_next_24h"] = df["rain"].iloc[::-1].rolling(window=24).sum().iloc[::-1]

df["target_rain_next_day"] = (df["rain_sum_next_24h"] > 0).astype(int)

In [41]:
df['temp_lag_1'] = df['temperature_2m'].shift(1)
df['temp_lag_3'] = df['temperature_2m'].shift(3)
df['temp_lag_6'] = df['temperature_2m'].shift(6)
df['temp_lag_12'] = df['temperature_2m'].shift(12)
df['temp_lag_24'] = df['temperature_2m'].shift(24)

df['wind_lag_1'] = df['wind_speed_10m'].shift(1)
df['rain_lag_1'] = df['rain'].shift(1)
df['pressure_lag_1'] = df['surface_pressure'].shift(1)

In [42]:
df['temp_roll_mean_6'] = df['temperature_2m'].rolling(6).mean()
df['temp_roll_mean_24'] = df['temperature_2m'].rolling(24).mean()
df['temp_roll_std_24'] = df['temperature_2m'].rolling(24).std()

df['wind_roll_mean_6'] = df['wind_speed_10m'].rolling(6).mean()
df['rain_roll_sum_24'] = df['rain'].rolling(24).sum()


In [13]:
df=df.dropna()
df.head()

,timestamp,temperature_2m,relative_humidity_2m,rain,surface_pressure,precipitation,wind_speed_10m,wind_direction_10m,apparent_temperature,cloud_cover,...,temp_delta,pressure_delta,wind_delta,rain_delta,wind_dir_sin,wind_dir_cos,cloud_binary,cloud_roll_mean_6,rain_sum_next_24h,target_rain_next_day
72,2026-05-04 00:00:00,12.00,48.792694,0.0,1004.93,0.0,6.55,164.054535,9.17,0.0,...,-1.80,-0.19,-2.85,0.0,0.274722,-0.961524,0,0.0,0.0,0
73,2026-05-04 01:00:00,10.60,53.150288,0.0,1004.86,0.0,5.82,158.198532,7.86,0.0,...,-1.40,-0.07,-0.73,0.0,0.371392,-0.928476,0,0.0,0.0,0
74,2026-05-04 02:00:00,9.85,56.080723,0.0,1004.93,0.0,5.12,161.564957,7.22,0.0,...,-0.75,0.07,-0.70,0.0,0.316229,-0.948683,0,0.0,0.0,0
75,2026-05-04 03:00:00,8.95,60.653603,0.0,1004.88,0.0,4.21,160.016800,6.50,0.0,...,-0.90,-0.05,-0.91,0.0,0.341745,-0.939793,0,0.0,0.0,0
76,2026-05-04 04:00:00,9.05,61.550327,0.0,1005.18,0.0,3.64,159.775055,6.73,0.0,...,0.10,0.30,-0.57,0.0,0.345707,-0.938343,0,0.0,0.0,0


In [14]:
table_name = "weather_logs_features"
df.to_sql(
    name=table_name,
    con=engine,
    if_exists="replace",  # ha már létezik a tábla, írja felül (futtathatod többször is)
    index=False,           # ne mentse el a pandas indexet külön oszlopként
    method="multi"         # felgyorsítja a nagy mennyiségű adat beszúrását
)

3162